# مرشّح الترددات المنخفضة (Low-Pass Filter)

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1 (only one subject for speed)

---

## What this notebook does

A low-pass filter removes **high-frequency components** (above 40 Hz) from the EEG signal. This eliminates:

- Power line interference (50/60 Hz)
- Muscle artifacts (EMG, which is high-frequency)
- Random high-frequency noise

We apply this **after** the high-pass filter, so the signal already has no DC offset or drift.

## What you should expect to see

After filtering, the signal should look **smoother** — the rapid jagged oscillations from the high-pass output should disappear. The signal retains the brain-relevant frequencies (1 to 40 Hz: delta, theta, alpha, beta).

## Key parameters

| Parameter | Value | Meaning |
|-----------|-------|---------|
| Cutoff | 40 Hz | Frequencies above 40 Hz are removed |
| Order | 4 | Steepness of the filter roll-off |
| Method | Butterworth | Flat response in the passband |
| filtfilt | Yes | Zero-phase (no time delay) |

## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. Clone the resources repo and download one subject

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

## 3. Load the EEG signal and apply high-pass first

We first apply the high-pass filter (1 Hz) to remove drift, then apply the low-pass filter (40 Hz) to remove noise. This two-step process is equivalent to a band-pass filter, which we will see in the next notebook.

In [ ]:
import numpy as np
from scipy import signal
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

# Step 1: High-pass filter at 1 Hz
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False)
    return signal.filtfilt(b, a, data)

# Step 2: Low-pass filter at 40 Hz
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    return signal.filtfilt(b, a, data)

filtered_hp = butter_highpass_filter(channel_data, cutoff=1.0, fs=fs)
filtered_lp = butter_lowpass_filter(filtered_hp, cutoff=40.0, fs=fs)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

## 4. Interactive plot: raw, high-pass, then low-pass

Three subplots show the full progression:

1. **Raw EEG** (top) — drift + noise
2. **After high-pass** (middle) — drift removed, noise remains
3. **After low-pass** (bottom) — drift removed AND noise removed

**What to look for:**
- The raw signal has a slow drift (visible as a gradual trend)
- The high-pass signal is centered around zero but still jagged
- The low-pass signal is centered around zero AND smooth
- The brain-relevant oscillations (slow waves) are preserved in all three

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Raw EEG (P4)', 'After high-pass (1 Hz)', 'After low-pass (40 Hz)'))

fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot],
                         name='Raw', line=dict(color='gray', width=0.5)),
               row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_hp[:n_plot],
                         name='High-pass', line=dict(color='green', width=0.5)),
               row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_lp[:n_plot],
                         name='HP + LP', line=dict(color='red', width=0.5)),
               row=3, col=1)

fig.update_layout(height=800, title_text='Low-Pass Filter: Raw → High-pass → Low-pass',
                  xaxis3_title='Time (s)', yaxis_title='EEG (uV)',
                  yaxis2_title='EEG (uV)', yaxis3_title='EEG (uV)')
fig.show()

## 5. What did we learn?

- The low-pass filter at 40 Hz **removed high-frequency noise** from the signal
- The signal is now **smoother** while preserving the brain-relevant frequencies (1 to 40 Hz)
- Combined with the high-pass filter, we now have a clean signal in the **1 to 40 Hz band**
- The next notebook shows how to do this in a **single step** with a band-pass filter